# PAC Plots: siganl vs background
This notebook contains calculations / plots for September 2026 PAC. The goal is to generate simulated signal over backgroudnd chart for run1a/run1b scenarios. 1809 keV, and then 347 keV photons are focused

In [1]:
from __future__ import print_function
import sys, os
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import ROOT
from ROOT import gROOT, gStyle, gDirectory, gPad

import filepath 
import portROOT2pd
from pdgid import pdgid_dict
import plot_utils
import constants

In [5]:
nPOT = 1.0e12

# run1a/b, all to VD101
event_dict = {
    "MuBeam1a":{
        "file": "tmp/mu101_MDC2025ai_dump.root",
        "nEvent": 3642.,
        "weight": 1./1000000000.*213816./100000000.*nPOT, # multiple event # by this
    },
    "EleBeam1a":{
        "file": "tmp/ele101_MDC2025ai_dump.root",
        "nEvent": 37913.,
        "weight": 1./5441845688.*5532579./100000000.*nPOT, # multiple event # by this
    },
    "Neutrals1a":{
        "file": "tmp/neutrals101_MDC2025ag_dump.root",
        "nEvent": 80.,        
        "weight": 1./1983800000.*53426053./100000000.*nPOT, # multiple event # by this
    },
    # 1809 dataset not present for now. Use TargetStops muonstops to generate photons, 
    # number based on solid angle, timing based on capture time, decay time (864.0 ns), and time of flight
    "MuBeam1b":{
        "file": "tmp/mu101_Run1Ban001_dump.root",
        "nEvent": 1366.,
        "weight": 1./897800000.*25584287./2000000000.*nPOT, # multiple event # by this
    },
    "EleBeam1b":{
        "file": "tmp/ele101_Run1Ban001_dump.root",
        "nEvent": 2297.,
        "weight": 1./999600000.*172423754./2000000000.*nPOT, # multiple event # by this
    },
    "1809Beam1b":{
         "file": "tmp/tgtstps101_Run1Ban001_dump.root",
        "nEvent": 105.,
        "weight": 1./1000000000.*79278./2000000000.*1000.*nPOT, # multiple event # by this
    },
}

for k, v in event_dict.items():
    print (k, v["weight"])

MuBeam1a 2.13816
EleBeam1a 10.166732607284471
Neutrals1a 269.31168968646034
MuBeam1b 14.248322009356206
EleBeam1b 86.2463755502201
1809Beam1b 39.639


In [ ]:
# For target stops cat, in run1a expected stopped muon number
nMuStopped1a = 1442670./4000000000.*213816./100000000.*1000.*nPOT
n1089 = nMuStopped1a*0.61*0.51
# solid angle extrapolation
def n1809atVD101(radius): #in mm
    r = constants.VD101_Z-(constants.stopping_target_start_Z+constants.stopping_target_end_Z)/2.
    solidangleratio = radius*radius/4/r/r
    return n1089*solidangleratio

In [ ]:
# first HPGe
hole_radius = constants.hole1_R 
# now construct 1809 keV plot. Create a ROOT TH1D, with the range of 1809+/-20. 

# add signals: fill in the TH1D 
# frist determine number of signals to add. that would be n1809atVD101(hole_radius),
# then generate this many random numbers from a Gaussian distribution with mean 1809, 
# sigma 2.5 keV, and fill in the TH1D.

# add background to the THStack. These coming from "MuBeam1a","EleBeam1a","Neutrals1a"
# open the corresponding ROOT files, get the TTree, use RDataFrame to go through entries
# filter entries that have 